In [2]:
import json
import pandas as pd
from IPython.display import display

# Load the dataset
with open('data/data_2018.json') as f:
    raw_data = json.load(f)

# The JSON contains multiple keys with arrays of different lengths — flatten each into its own DataFrame
data_stadiums = pd.DataFrame(raw_data['stadiums'])
data_teams = pd.DataFrame(raw_data['teams'])
data_tv = pd.DataFrame(raw_data['tvchannels'])

# ── Cleaning data_2018 ──────────────────────────────────────────────────────

# stadiums & teams : aucun nettoyage nécessaire (pas de nulls, pas de doublons,
#                    types corrects)

# tvchannels : la colonne 'lang' contient des listes Python (ex. ['da'])
#              → aplatir en chaîne de caractères séparée par des virgules
data_tv['lang'] = data_tv['lang'].apply(
    lambda v: ', '.join(v) if isinstance(v, list) else str(v)
)

# ── Chargement des CSV ──────────────────────────────────────────────────────
matches = pd.read_csv('data/matches_19302010.csv')
world_cup_matches = pd.read_csv('data/WorldCupMatches2014.csv')

# Display the first few rows
print("Stadiums:")
display(data_stadiums.head())
print("\nTeams:")
display(data_teams.head())
print("\nTV Channels:")
display(data_tv.head())

# Check for missing values
print("\nMissing values in matches:")
display(matches.isnull().sum())


Stadiums:


,id,name,city,lat,lng,image
0,1,Luzhniki Stadium,Moscow,55.715765,37.551522,https://upload.wikimedia.org/wikipedia/commons...
1,2,Otkrytiye Arena,Moscow,55.817765,37.440363,https://upload.wikimedia.org/wikipedia/commons...
2,3,Krestovsky Stadium,Saint Petersburg,59.972740,30.221408,https://upload.wikimedia.org/wikipedia/commons...
3,4,Kaliningrad Stadium,Kaliningrad,54.698157,20.533859,https://upload.wikimedia.org/wikipedia/commons...
4,5,Kazan Arena,Kazan,55.820983,49.160966,https://upload.wikimedia.org/wikipedia/commons...



Teams:


,id,name,fifaCode,iso2,flag,emoji,emojiString
0,1,Russia,RUS,ru,https://upload.wikimedia.org/wikipedia/en/thum...,flag-ru,🇷🇺
1,2,Saudi Arabia,KSA,sa,https://upload.wikimedia.org/wikipedia/commons...,flag-sa,🇸🇦
2,3,Egypt,EGY,eg,https://upload.wikimedia.org/wikipedia/commons...,flag-eg,🇪🇬
3,4,Uruguay,URU,uy,https://upload.wikimedia.org/wikipedia/commons...,flag-uy,🇺🇾
4,5,Portugal,POR,pt,https://upload.wikimedia.org/wikipedia/commons...,flag-pt,🇵🇹



TV Channels:


,id,name,icon,country,iso2,lang
0,1,DR1,https://upload.wikimedia.org/wikipedia/commons...,Denmark,dk,da
1,2,TV2,https://upload.wikimedia.org/wikipedia/commons...,Denmark,dk,da
2,3,BBC UK,https://upload.wikimedia.org/wikipedia/commons...,UK,en,en
3,4,ITV UK,https://upload.wikimedia.org/wikipedia/en/9/92...,UK,en,en
4,5,ITV4 UK,https://upload.wikimedia.org/wikipedia/en/9/92...,UK,en,en



Missing values in matches:


edition    0
round      0
score      0
team1      0
team2      0
url        0
venue      0
year       0
dtype: int64

In [3]:
matches = pd.read_csv('data/matches_19302010.csv')
wold_cup_matches = pd.read_csv('data/WorldCupMatches2014.csv', delimiter=';')

display(matches.head())
display(wold_cup_matches.head())

,edition,round,score,team1,team2,url,venue,year
0,1930-URUGUAY,GROUP_STAGE,4-1 (3-0),France,Mexico (México),1930_URUGUAY_FS.htm#1-WC-30-I,Montevideo.,1930
1,1930-URUGUAY,GROUP_STAGE,3-0 (2-0),USA,Belgium (België),1930_URUGUAY_FS.htm#13-WC-30-I,Montevideo.,1930
2,1930-URUGUAY,GROUP_STAGE,2-1 (2-0),Yugoslavia (Југославија),Brazil (Brasil),1930_URUGUAY_FS.htm#7-WC-30-I,Montevideo.,1930
3,1930-URUGUAY,GROUP_STAGE,3-1 (1-0),Romania (România),Peru (Perú),1930_URUGUAY_FS.htm#10-WC-30-I,Montevideo.,1930
4,1930-URUGUAY,GROUP_STAGE,1-0 (0-0),Argentina,France,1930_URUGUAY_FS.htm#2-WC-30-I,Montevideo.,1930


,Year,Datetime,Stage,Stadium,City,Home Team Name,Home Team Goals,Away Team Goals,Away Team Name,Win conditions,Attendance,Half-time Home Goals,Half-time Away Goals,Referee,Assistant 1,Assistant 2,RoundID,MatchID,Home Team Initials,Away Team Initials
0,2014,12 Jun 2014 - 17:00,Group A,Arena de Sao Paulo,Sao Paulo,Brazil,3,1,Croatia,,62103.0,1,1,NISHIMURA Yuichi (JPN),SAGARA Toru (JPN),NAGI Toshiyuki (JPN),255931,300186456,BRA,CRO
1,2014,13 Jun 2014 - 13:00,Group A,Estadio das Dunas,Natal,Mexico,1,0,Cameroon,,39216.0,0,0,ROLDAN Wilmar (COL),CLAVIJO Humberto (COL),DIAZ Eduardo (COL),255931,300186492,MEX,CMR
2,2014,13 Jun 2014 - 16:00,Group B,Arena Fonte Nova,Salvador,Spain,1,5,Netherlands,,48173.0,1,1,Nicola RIZZOLI (ITA),Renato FAVERANI (ITA),Andrea STEFANI (ITA),255931,300186510,ESP,NED
3,2014,13 Jun 2014 - 18:00,Group B,Arena Pantanal,Cuiaba,Chile,3,1,Australia,,40275.0,2,1,Noumandiez DOUE (CIV),YEO Songuifolo (CIV),BIRUMUSHAHU Jean Claude (BDI),255931,300186473,CHI,AUS
4,2014,14 Jun 2014 - 13:00,Group C,Estadio Mineirao,Belo Horizonte,Colombia,3,0,Greece,,57174.0,1,0,GEIGER Mark (USA),HURD Sean (USA),FLETCHER Joe (CAN),255931,300186471,COL,GRE


In [4]:
import re

# ─────────────────────────────────────────────
# Cleaning : matches_19302010
# ─────────────────────────────────────────────

matches_clean = matches.copy()

# 1. Noms d'équipes : supprimer la traduction entre parenthèses
#    ex. "Brazil (Brasil)" → "Brazil"
def clean_team_name(name: str) -> str:
    return re.sub(r'\s*\(.*?\)', '', str(name)).strip()

matches_clean['team1'] = matches_clean['team1'].apply(clean_team_name)
matches_clean['team2'] = matches_clean['team2'].apply(clean_team_name)

# 2. Villes : supprimer le point final
#    ex. "Montevideo." → "Montevideo"
matches_clean['venue'] = matches_clean['venue'].str.rstrip('.').str.strip()

# 3. Score : séparer le score final et le score à la mi-temps
#    ex. "4-1 (3-0)" → full_time="4-1", half_time="3-0"
score_split = matches_clean['score'].str.extract(r'(?P<full_time>\d+-\d+)\s*\((?P<half_time>\d+-\d+)\)')
matches_clean[['full_time_score', 'half_time_score']] = score_split

# 4. Doublons
before = len(matches_clean)
matches_clean.drop_duplicates(inplace=True)
print(f"matches — doublons supprimés : {before - len(matches_clean)}")

# 5. Valeurs manquantes
matches_clean.fillna(matches_clean.select_dtypes(include='number').mean(), inplace=True)

print("matches_clean :")
display(matches_clean.head())


# ─────────────────────────────────────────────
# Cleaning : WorldCupMatches2014
# ─────────────────────────────────────────────

wc_clean = wold_cup_matches.copy()

# 1. Colonnes texte : supprimer les espaces en trop (pandas >= 3 : 'str' au lieu de 'object')
str_cols = wc_clean.select_dtypes(include='str').columns
wc_clean[str_cols] = wc_clean[str_cols].apply(lambda col: col.str.strip())

# 2. Noms d'équipes (cohérence de casse)
wc_clean['Home Team Name'] = wc_clean['Home Team Name'].str.title()
wc_clean['Away Team Name'] = wc_clean['Away Team Name'].str.title()

# 3. Villes
wc_clean['City'] = wc_clean['City'].str.title()

# 4. Format de date : "12 Jun 2014 - 17:00" → datetime
wc_clean['Datetime'] = pd.to_datetime(
    wc_clean['Datetime'].str.strip(),
    format='%d %b %Y - %H:%M'
)

# 5. Valeurs manquantes : Attendance (2 lignes) — évite ChainedAssignmentError
wc_clean['Attendance'] = wc_clean['Attendance'].fillna(wc_clean['Attendance'].median())

# 6. Doublons
before = len(wc_clean)
wc_clean.drop_duplicates(inplace=True)
print(f"world_cup_matches — doublons supprimés : {before - len(wc_clean)}")

print("wc_clean :")
display(wc_clean.head())
display(wc_clean.dtypes)


matches — doublons supprimés : 0
matches_clean :


,edition,round,score,team1,team2,url,venue,year,full_time_score,half_time_score
0,1930-URUGUAY,GROUP_STAGE,4-1 (3-0),France,Mexico,1930_URUGUAY_FS.htm#1-WC-30-I,Montevideo,1930,4-1,3-0
1,1930-URUGUAY,GROUP_STAGE,3-0 (2-0),USA,Belgium,1930_URUGUAY_FS.htm#13-WC-30-I,Montevideo,1930,3-0,2-0
2,1930-URUGUAY,GROUP_STAGE,2-1 (2-0),Yugoslavia,Brazil,1930_URUGUAY_FS.htm#7-WC-30-I,Montevideo,1930,2-1,2-0
3,1930-URUGUAY,GROUP_STAGE,3-1 (1-0),Romania,Peru,1930_URUGUAY_FS.htm#10-WC-30-I,Montevideo,1930,3-1,1-0
4,1930-URUGUAY,GROUP_STAGE,1-0 (0-0),Argentina,France,1930_URUGUAY_FS.htm#2-WC-30-I,Montevideo,1930,1-0,0-0


world_cup_matches — doublons supprimés : 16
wc_clean :


,Year,Datetime,Stage,Stadium,City,Home Team Name,Home Team Goals,Away Team Goals,Away Team Name,Win conditions,Attendance,Half-time Home Goals,Half-time Away Goals,Referee,Assistant 1,Assistant 2,RoundID,MatchID,Home Team Initials,Away Team Initials
0,2014,2014-06-12 17:00:00,Group A,Arena de Sao Paulo,Sao Paulo,Brazil,3,1,Croatia,,62103.0,1,1,NISHIMURA Yuichi (JPN),SAGARA Toru (JPN),NAGI Toshiyuki (JPN),255931,300186456,BRA,CRO
1,2014,2014-06-13 13:00:00,Group A,Estadio das Dunas,Natal,Mexico,1,0,Cameroon,,39216.0,0,0,ROLDAN Wilmar (COL),CLAVIJO Humberto (COL),DIAZ Eduardo (COL),255931,300186492,MEX,CMR
2,2014,2014-06-13 16:00:00,Group B,Arena Fonte Nova,Salvador,Spain,1,5,Netherlands,,48173.0,1,1,Nicola RIZZOLI (ITA),Renato FAVERANI (ITA),Andrea STEFANI (ITA),255931,300186510,ESP,NED
3,2014,2014-06-13 18:00:00,Group B,Arena Pantanal,Cuiaba,Chile,3,1,Australia,,40275.0,2,1,Noumandiez DOUE (CIV),YEO Songuifolo (CIV),BIRUMUSHAHU Jean Claude (BDI),255931,300186473,CHI,AUS
4,2014,2014-06-14 13:00:00,Group C,Estadio Mineirao,Belo Horizonte,Colombia,3,0,Greece,,57174.0,1,0,GEIGER Mark (USA),HURD Sean (USA),FLETCHER Joe (CAN),255931,300186471,COL,GRE


Year                             int64
Datetime                datetime64[us]
Stage                              str
Stadium                            str
City                               str
Home Team Name                     str
Home Team Goals                  int64
Away Team Goals                  int64
Away Team Name                     str
Win conditions                     str
Attendance                     float64
Half-time Home Goals             int64
Half-time Away Goals             int64
Referee                            str
Assistant 1                        str
Assistant 2                        str
RoundID                          int64
MatchID                          int64
Home Team Initials                 str
Away Team Initials                 str
dtype: object

In [5]:
import duckdb
con = duckdb.connect('foot_etl.duckdb')
con.execute("SELECT * FROM stg_matches LIMIT 10").fetchdf()

,edition,round,score,team1,team2,venue,year,url,full_time_score,half_time_score
0,1930-URUGUAY,GROUP_STAGE,4-1 (3-0),France,Mexico,Montevideo,1930,1930_URUGUAY_FS.htm#1-WC-30-I,4-1,3-0
1,1930-URUGUAY,GROUP_STAGE,3-0 (2-0),USA,Belgium,Montevideo,1930,1930_URUGUAY_FS.htm#13-WC-30-I,3-0,2-0
2,1930-URUGUAY,GROUP_STAGE,2-1 (2-0),Yugoslavia,Brazil,Montevideo,1930,1930_URUGUAY_FS.htm#7-WC-30-I,2-1,2-0
3,1930-URUGUAY,GROUP_STAGE,3-1 (1-0),Romania,Peru,Montevideo,1930,1930_URUGUAY_FS.htm#10-WC-30-I,3-1,1-0
4,1930-URUGUAY,GROUP_STAGE,1-0 (0-0),Argentina,France,Montevideo,1930,1930_URUGUAY_FS.htm#2-WC-30-I,1-0,0-0
5,1930-URUGUAY,GROUP_STAGE,3-0 (1-0),Chile,Mexico,Montevideo,1930,1930_URUGUAY_FS.htm#3-WC-30-I,3-0,1-0
6,1930-URUGUAY,GROUP_STAGE,4-0 (0-0),Yugoslavia,Bolivia,Montevideo,1930,1930_URUGUAY_FS.htm#8-WC-30-I,4-0,0-0
7,1930-URUGUAY,GROUP_STAGE,3-0 (2-0),USA,Paraguay,Montevideo,1930,1930_URUGUAY_FS.htm#14-WC-30-I,3-0,2-0
8,1930-URUGUAY,GROUP_STAGE,1-0 (0-0),Uruguay,Peru,Montevideo,1930,1930_URUGUAY_FS.htm#11-WC-30-I,1-0,0-0
9,1930-URUGUAY,GROUP_STAGE,1-0 (0-0),Chile,France,Montevideo,1930,1930_URUGUAY_FS.htm#4-WC-30-I,1-0,0-0


In [ ]:
with open('data/worldcup.json') as f:
    raw_wc2022 = json.load(f)

# The JSON has two keys: 'name' (str) and 'matches' (list of match dicts)
# Each match has nested 'score': {'ft': [g1, g2], 'ht': [g1, g2]} → flatten
wc2022 = pd.DataFrame(raw_wc2022['matches'])

# Flatten nested score dict into separate columns
wc2022['score_ft_home'] = wc2022['score'].apply(lambda s: s['ft'][0])
wc2022['score_ft_away'] = wc2022['score'].apply(lambda s: s['ft'][1])
wc2022['score_ht_home'] = wc2022['score'].apply(lambda s: s['ht'][0] if 'ht' in s else None)
wc2022['score_ht_away'] = wc2022['score'].apply(lambda s: s['ht'][1] if 'ht' in s else None)
wc2022.drop(columns=['score', 'goals1', 'goals2'], inplace=True)

# Parse date
wc2022['date'] = pd.to_datetime(wc2022['date'])

print(f"World Cup 2022 — {len(wc2022)} matches")
display(wc2022.head())
display(wc2022.dtypes)

World Cup 2022 — 64 matches


,round,date,time,team1,team2,group,ground,score_ft_home,score_ft_away,score_ht_home,score_ht_away
0,Matchday 1,2022-11-20,19:00,Qatar,Ecuador,Group A,"Al Bayt Stadium, Al Khor",0,2,0.0,2.0
1,Matchday 2,2022-11-21,19:00,Senegal,Netherlands,Group A,"Al Thumama Stadium, Doha",0,2,0.0,0.0
2,Matchday 6,2022-11-25,16:00,Qatar,Senegal,Group A,"Al Thumama Stadium, Doha",1,3,0.0,1.0
3,Matchday 6,2022-11-25,19:00,Netherlands,Ecuador,Group A,"Khalifa International Stadium, Al Rayyan",1,1,1.0,0.0
4,Matchday 10,2022-11-29,18:00,Ecuador,Senegal,Group A,"Khalifa International Stadium, Al Rayyan",1,2,0.0,1.0


round                       str
date             datetime64[us]
time                        str
team1                       str
team2                       str
group                       str
ground                      str
score_ft_home             int64
score_ft_away             int64
score_ht_home           float64
score_ht_away           float64
dtype: object

In [16]:
wc2022

,round,date,time,team1,team2,group,ground,score_ft_home,score_ft_away,score_ht_home,score_ht_away
0,Matchday 1,2022-11-20,19:00,Qatar,Ecuador,Group A,"Al Bayt Stadium, Al Khor",0,2,0.0,2.0
1,Matchday 2,2022-11-21,19:00,Senegal,Netherlands,Group A,"Al Thumama Stadium, Doha",0,2,0.0,0.0
2,Matchday 6,2022-11-25,16:00,Qatar,Senegal,Group A,"Al Thumama Stadium, Doha",1,3,0.0,1.0
3,Matchday 6,2022-11-25,19:00,Netherlands,Ecuador,Group A,"Khalifa International Stadium, Al Rayyan",1,1,1.0,0.0
4,Matchday 10,2022-11-29,18:00,Ecuador,Senegal,Group A,"Khalifa International Stadium, Al Rayyan",1,2,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
59,Quarter-finals,2022-12-10,22:00,England,France,NaN,"Al Bayt Stadium, Al Khor",1,2,0.0,1.0
60,Semi-finals,2022-12-13,22:00,Argentina,Croatia,NaN,"Lusail Iconic Stadium, Lusail",3,0,2.0,0.0
61,Semi-finals,2022-12-14,22:00,France,Morocco,NaN,"Al Bayt Stadium, Al Khor",2,0,1.0,0.0
62,Match for third place,2022-12-17,18:00,Croatia,Morocco,NaN,"Khalifa International Stadium, Al Rayyan",2,1,2.0,1.0
